**A) gold_trip_features (Trip-level feature store)**
- Purpose: canonical trip features used for rollups + explainability.
- Grain: 1 row per Trip_ID


In [1]:
%%sql
CREATE TABLE IF NOT EXISTS gold_trip_features (
  trip_id                    STRING,
  policyholder_id            BIGINT,
  vin                        STRING,

  start_time                 TIMESTAMP,
  end_time                   TIMESTAMP,
  trip_date                  DATE,
  time_of_day_category       STRING,          -- Morning/Afternoon/Evening/Night
  duration_min               DOUBLE,
  distance_miles             DOUBLE,

  avg_speed                  DOUBLE,
  peak_high_speed            DOUBLE,
  peak_low_speed             DOUBLE,

  speeding_incidents         INT,
  max_speed_limit_exceeded   DOUBLE,

  sudden_braking_count       INT,
  rapid_acceleration_count   INT,
  harsh_cornering_count      INT,

  safety_score               DOUBLE,
  braking_index              DOUBLE,
  acceleration_index         DOUBLE,
  trip_risk_level            STRING,          -- Low/Medium/High

  -- Derived “UBI features”
  is_night                   BOOLEAN,
  harsh_events_total         INT,
  harsh_events_per_100_miles DOUBLE,
  speeding_per_100_miles     DOUBLE,

  record_source              STRING,
  ingest_ts                  TIMESTAMP
)
USING DELTA;


StatementMeta(, cef8c836-a29c-49ef-8928-c1e16a9e6524, 2, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

**B) gold_driver_monthly_features (Driver-month rollup)**
- Purpose: core UBI driver behavior profile used for risk & pricing.
- Grain: 1 row per (policyholder_id, feature_month)


In [3]:
%%sql
CREATE TABLE IF NOT EXISTS gold_driver_monthly_features (
  policyholder_id              BIGINT,
  feature_month                DATE,            -- first day of month

  trips_count                  BIGINT,
  total_miles                  DOUBLE,
  total_duration_min           DOUBLE,

  night_trips_count            BIGINT,
  night_miles                  DOUBLE,
  night_miles_share            DOUBLE,

  avg_safety_score             DOUBLE,
  p10_safety_score             DOUBLE,
  p90_safety_score             DOUBLE,

  total_speeding_incidents     BIGINT,
  speeding_per_100_miles       DOUBLE,

  total_harsh_braking          BIGINT,
  total_rapid_acceleration     BIGINT,
  total_harsh_cornering        BIGINT,
  harsh_events_total           BIGINT,
  harsh_events_per_100_miles   DOUBLE,

  high_risk_trip_share         DOUBLE,          -- % trips Trip_Risk_Level=High
  medium_risk_trip_share       DOUBLE,
  low_risk_trip_share          DOUBLE,

  feature_version              STRING,
  computed_ts                  TIMESTAMP
)
USING DELTA;


StatementMeta(, cef8c836-a29c-49ef-8928-c1e16a9e6524, 3, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

**C) gold_policy_period_features (Policy-period rollup)**
- Purpose: pricing-ready feature set aligned to actual policy term.
- Grain: 1 row per (policy_number, policy_start_date, policy_end_date)

In [4]:
%%sql
CREATE TABLE IF NOT EXISTS gold_policy_period_features (
  policy_number                STRING,
  policyholder_id              BIGINT,
  vehicle_vin                  STRING,
  coverage_type                STRING,

  policy_start_date            DATE,
  policy_end_date              DATE,
  current_premium              DOUBLE,

  total_trips                  BIGINT,
  total_miles                  DOUBLE,
  night_miles_share            DOUBLE,
  avg_safety_score             DOUBLE,

  speeding_per_100_miles       DOUBLE,
  harsh_events_per_100_miles   DOUBLE,
  high_risk_trip_share         DOUBLE,

  -- Optional enrichments
  vehicle_year                 INT,
  vehicle_make                 STRING,
  vehicle_model                STRING,

  feature_version              STRING,
  computed_ts                  TIMESTAMP
)
USING DELTA;

StatementMeta(, cef8c836-a29c-49ef-8928-c1e16a9e6524, 4, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

**D) gold_policy_period_loss (Outcome for calibration/validation)**
- Purpose: build the “truth” needed to calibrate expected loss cost.
- Grain: 1 row per policy period

In [5]:
%%sql
CREATE TABLE IF NOT EXISTS gold_policy_period_loss (
  policy_number           STRING,
  policy_start_date       DATE,
  policy_end_date         DATE,

  claims_count            BIGINT,
  total_claim_amount      DOUBLE,
  total_payout_amount     DOUBLE,

  accidents_count         BIGINT,
  high_severity_accidents BIGINT,

  label_version           STRING,
  computed_ts             TIMESTAMP
)
USING DELTA;

StatementMeta(, cef8c836-a29c-49ef-8928-c1e16a9e6524, 5, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

**E) gold_expected_loss_scores (Model output or rules-based output)**
1. Purpose: store expected loss cost and risk score.
2. Grain: 1 row per policy period

In [7]:
%%sql
CREATE TABLE IF NOT EXISTS gold_expected_loss_scores (
  policy_number           STRING,
  policy_start_date       DATE,
  policy_end_date         DATE,

  risk_score              DOUBLE,     -- 0..1 or 0..100
  expected_loss_cost      DOUBLE,     -- $ expected payout for next term (or same term)
  model_name              STRING,
  model_version           STRING,
  feature_version         STRING,

  scoring_ts              TIMESTAMP
)
USING DELTA;

StatementMeta(, cef8c836-a29c-49ef-8928-c1e16a9e6524, 6, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

**F) gold_policy_premium_recommendation (Premium amount output)**
- Purpose: the “product” table you serve to BI / downstream systems.
- Grain: 1 row per policy period

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS gold_policy_premium_recommendation (
  policy_number            STRING,
  policy_start_date        DATE,
  policy_end_date          DATE,
  coverage_type            STRING,

  current_premium          DOUBLE,
  expected_loss_cost       DOUBLE,

  target_loss_ratio        DOUBLE,
  expense_load             DOUBLE,
  profit_load              DOUBLE,

  recommended_technical_premium DOUBLE,  -- before guardrails
  recommended_premium      DOUBLE,       -- after caps/smoothing
  premium_change_pct       DOUBLE,

  actual_loss_ratio        DOUBLE,       -- gold_policy_period_loss.total_payout_amount / recommended_premium
  underpriced_flag         STRING,       -- "Y" if (indicated_premium / recommended_premium) > 1, else "N"
                                         -- where indicated_premium = (actual_loss_ratio / target_loss_ratio) - 1

  cap_applied_flag         BOOLEAN,
  smoothing_applied_flag   BOOLEAN,

  reason_codes             STRING,       -- e.g., JSON string
  scoring_ts               TIMESTAMP,

  model_version            STRING,
  feature_version          STRING
)
USING DELTA;

StatementMeta(, cef8c836-a29c-49ef-8928-c1e16a9e6524, 7, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

**G) gold_premium_reason_codes (Explainability, optional but powerful)**
- Purpose: store top drivers behind the recommendation (great for regulators & business trust).

In [9]:
%%sql
CREATE TABLE IF NOT EXISTS gold_premium_reason_codes (
  policy_number        STRING,
  policy_start_date    DATE,
  policy_end_date      DATE,

  factor_name          STRING,   -- e.g. "speeding_per_100_miles"
  factor_value         DOUBLE,
  direction            STRING,   -- "increased" / "decreased"
  contribution_score   DOUBLE,   -- relative importance
  computed_ts          TIMESTAMP
)
USING DELTA;

StatementMeta(, cef8c836-a29c-49ef-8928-c1e16a9e6524, 8, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>